# QuickPay FinTech Operations — Python Pipeline
**Parts 3 & 4 | Reconciliation + JSON Normalization**

This notebook covers:
- Part 3: Ledger vs Gateway reconciliation
- Part 4: API JSON normalization
- Dashboard support CSV generation

In [ ]:
import pandas as pd
import numpy as np
import json

# File paths
RAW = '01_data/raw/'
PROC = '01_data/processed/'
print('Libraries loaded. Ready to run pipeline.')

## Part 3: Reconciliation Workflow
### Step 1: Load Files

In [ ]:
ledger  = pd.read_csv(RAW + 'ledger.csv')
gateway = pd.read_csv(RAW + 'gateway.csv')

print(f'Ledger rows: {len(ledger)}')
print(f'Gateway rows: {len(gateway)}')
print('\n--- Ledger ---')
display(ledger)
print('\n--- Gateway ---')
display(gateway)

### Step 2: Check Duplicates and Nulls

In [ ]:
print('=== DUPLICATES ===')
print(f'Ledger duplicates: {ledger.duplicated().sum()}')
print(f'Gateway duplicates: {gateway.duplicated().sum()}')

print('\n=== NULL VALUES - LEDGER ===')
print(ledger.isnull().sum())

print('\n=== NULL VALUES - GATEWAY ===')
print(gateway.isnull().sum())

# Drop duplicates if any
ledger.drop_duplicates(inplace=True)
gateway.drop_duplicates(inplace=True)
print('\nDeduplication complete.')

### Step 3: Records Missing in Gateway

In [ ]:
missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway['transaction_id'])].copy()
missing_in_gateway['issue'] = 'missing_in_gateway'
print(f'Transactions in Ledger but NOT in Gateway: {len(missing_in_gateway)}')
display(missing_in_gateway)
missing_in_gateway.to_csv(PROC + 'missing_in_gateway.csv', index=False)
print('Saved: missing_in_gateway.csv')

### Step 4: Records Missing in Ledger

In [ ]:
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger['transaction_id'])].copy()
missing_in_ledger['issue'] = 'missing_in_ledger'
print(f'Transactions in Gateway but NOT in Ledger: {len(missing_in_ledger)}')
display(missing_in_ledger)
missing_in_ledger.to_csv(PROC + 'missing_in_ledger.csv', index=False)
print('Saved: missing_in_ledger.csv')

### Step 5: Amount Mismatches

In [ ]:
merged = ledger.merge(gateway, on='transaction_id', suffixes=('_ledger','_gateway'))

amount_mis = merged[merged['amount_usd_ledger'] != merged['amount_usd_gateway']].copy()
amount_mis['amount_diff'] = amount_mis['amount_usd_ledger'] - amount_mis['amount_usd_gateway']
print(f'Amount mismatches: {len(amount_mis)}')
display(amount_mis[['transaction_id','amount_usd_ledger','amount_usd_gateway','amount_diff']])
amount_mis.to_csv(PROC + 'amount_mismatches.csv', index=False)
print('Saved: amount_mismatches.csv')

### Step 6: Status Mismatches

In [ ]:
status_mis = merged[merged['status_ledger'] != merged['status_gateway']].copy()
print(f'Status mismatches: {len(status_mis)}')
display(status_mis[['transaction_id','status_ledger','status_gateway']])
status_mis.to_csv(PROC + 'status_mismatches.csv', index=False)
print('Saved: status_mismatches.csv')

### Step 7: Final Reconciliation Report

In [ ]:
merged['reconciliation_status'] = 'matched'
merged.loc[merged['amount_usd_ledger'] != merged['amount_usd_gateway'], 'reconciliation_status'] = 'amount_mismatch'
status_mask = merged['status_ledger'] != merged['status_gateway']
merged.loc[status_mask & (merged['reconciliation_status']=='matched'), 'reconciliation_status'] = 'status_mismatch'
merged.loc[status_mask & (merged['reconciliation_status']=='amount_mismatch'), 'reconciliation_status'] = 'amount_and_status_mismatch'

recon = merged[['transaction_id','transaction_date_ledger','merchant_id_ledger',
                 'amount_usd_ledger','amount_usd_gateway',
                 'status_ledger','status_gateway','reconciliation_status']].copy()
recon.rename(columns={'transaction_date_ledger':'transaction_date',
                       'merchant_id_ledger':'merchant_id'}, inplace=True)
display(recon)
recon.to_csv(PROC + 'reconciliation_report.csv', index=False)
print('Saved: reconciliation_report.csv')

### Step 8: Summary Metrics

In [ ]:
issue_count = (len(merged[merged['reconciliation_status']!='matched'])
               + len(missing_in_gateway) + len(missing_in_ledger))

summary = {
    'total_ledger_rows': int(len(ledger)),
    'total_gateway_rows': int(len(gateway)),
    'missing_in_gateway_count': int(len(missing_in_gateway)),
    'missing_in_ledger_count': int(len(missing_in_ledger)),
    'amount_mismatch_count': int(len(amount_mis)),
    'status_mismatch_count': int(len(status_mis)),
    'reconciliation_issue_count': int(issue_count),
    'ledger_total_amount': round(float(ledger['amount_usd'].sum()), 2),
    'gateway_total_amount': round(float(gateway['amount_usd'].sum()), 2),
    'amount_at_risk': round(float(amount_mis['amount_usd_ledger'].sum()), 2)
}

with open('04_python/summary_metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary Metrics:')
print(json.dumps(summary, indent=2))
print('\nSaved: summary_metrics.json')

---
## Part 4: JSON Normalization
### Step 9: Load and Flatten api_response_sample.json

In [ ]:
with open(RAW + 'api_response_sample.json') as f:
    api_data = json.load(f)

print('API metadata:')
print(f'  Generated at: {api_data["generated_at"]}')
print(f'  Source: {api_data["source"]}')
print(f'  Total batches: {len(api_data["batches"])}')

# Flatten nested structure
rows = []
for batch in api_data['batches']:
    for s in batch['settlements']:
        rows.append({
            'batch_id': batch['batch_id'],
            'merchant_id': batch['merchant']['merchant_id'],
            'merchant_name': batch['merchant']['merchant_name'],
            'merchant_region': batch['merchant']['region'],
            'settlement_id': s['settlement_id'],
            'amount_usd': s['amount_usd'],
            'status': s['status'],
            'processed_at': s['processed_at'],
            'bank_name': s['bank']['name'],
            'bank_country': s['bank']['country'],
            'generated_at': api_data['generated_at'],
            'source': api_data['source']
        })

df_api = pd.DataFrame(rows)
print(f'\nFlattened rows: {len(df_api)}')
display(df_api)

### Step 10: Clean Column Names and Convert Datetimes

In [ ]:
# Clean column names — lowercase and snake_case
df_api.columns = [c.lower().replace(' ', '_') for c in df_api.columns]

# Convert datetime fields
df_api['processed_at'] = pd.to_datetime(df_api['processed_at'])
df_api['generated_at'] = pd.to_datetime(df_api['generated_at'])

# Add derived date column
df_api['processed_date'] = df_api['processed_at'].dt.date

print('Column names:', df_api.columns.tolist())
print('\nData types:')
print(df_api.dtypes)
display(df_api)

### Step 11: Save Normalized Output

In [ ]:
df_api.to_csv(PROC + 'api_normalized.csv', index=False)
print(f'Saved: api_normalized.csv ({len(df_api)} rows, {len(df_api.columns)} columns)')

---
## Dashboard Support CSVs
Generate summary files for Looker Studio

In [ ]:
cleaned = pd.read_csv(PROC + 'cleaned_transactions.csv')

# Daily summary
daily = cleaned.groupby('transaction_date').agg(
    total_gmv=('amount_usd','sum'),
    captured_gmv=('amount_usd', lambda x: x[cleaned.loc[x.index,'status']=='captured'].sum()),
    transaction_count=('transaction_id','count'),
    success_count=('status', lambda x: (x=='captured').sum()),
).reset_index()
daily['success_rate_pct'] = (daily['success_count']/daily['transaction_count']*100).round(2)
daily.to_csv(PROC + 'daily_summary.csv', index=False)

# Payment method breakdown
pm = cleaned.groupby('payment_method').agg(
    transaction_count=('transaction_id','count'),
    total_gmv=('amount_usd','sum'),
).reset_index()
pm.to_csv(PROC + 'payment_method_breakdown.csv', index=False)

# Region breakdown
rb = cleaned.groupby('gateway_region').agg(
    transaction_count=('transaction_id','count'),
    total_gmv=('amount_usd','sum'),
    avg_risk_score=('risk_score','mean'),
    chargeback_count=('status', lambda x: (x=='chargeback').sum()),
).reset_index()
rb['avg_risk_score'] = rb['avg_risk_score'].round(2)
rb.to_csv(PROC + 'region_breakdown.csv', index=False)

# Merchant performance summary
mp = cleaned.groupby(['merchant_id','merchant_name','gateway_region']).agg(
    total_txns=('transaction_id','count'),
    total_gmv=('amount_usd','sum'),
    captured_gmv=('amount_usd', lambda x: x[cleaned.loc[x.index,'status']=='captured'].sum()),
    avg_risk=('risk_score','mean'),
    chargeback_count=('status', lambda x: (x=='chargeback').sum()),
    high_risk_count=('high_risk_flag','sum'),
    high_value_count=('high_value_flag','sum'),
).reset_index()
mp['chargeback_ratio_pct'] = (mp['chargeback_count']/mp['total_txns']*100).round(2)
mp.to_csv(PROC + 'merchant_performance_summary.csv', index=False)

print('All dashboard CSVs saved.')
print('daily_summary.csv:', len(daily), 'rows')
print('payment_method_breakdown.csv:', len(pm), 'rows')
print('region_breakdown.csv:', len(rb), 'rows')
print('merchant_performance_summary.csv:', len(mp), 'rows')

---
## Pipeline Complete
All output files have been generated in `01_data/processed/`.
Reconciliation metrics saved to `04_python/summary_metrics.json`.